# European index training data
Split 13 Oxford–Man indices by calendar year: train 2003–2014, validation 2015–2016, test 2017–2019. Only training data are assessed and exported. Coverage uses each exchange’s scheduled full trading days. Missing values stay missing.

Requires pandas, NumPy and `exchange-calendars==4.13.3`. Run from this notebook’s folder or the repository root.

In [9]:
from pathlib import Path

import exchange_calendars as xcals
import numpy as np
import pandas as pd

root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'data/raw/oxford_man').is_dir())
markets = {'.AEX': 'XAMS', '.BFX': 'XBRU', '.BVLG': 'XLIS', '.FCHI': 'XPAR',
           '.FTMIB': 'XMIL', '.FTSE': 'XLON', '.GDAXI': 'XETR', '.IBEX': 'XMAD',
           '.OMXC20': 'XCSE', '.OMXHPI': 'XHEL', '.OMXSPI': 'XSTO', '.OSEAX': 'XOSL', '.SSMI': 'XSWX'}

path = root / 'data/raw/oxford_man/oxfordmanrealizedvolatilityindices_20261109.csv'

panel = pd.read_csv(path).rename(columns={'Unnamed: 0': 'date'})
panel = panel.loc[panel.Symbol.isin(markets)].copy()
panel['date'] = pd.to_datetime(panel.date.str[:10])
panel = panel.set_index(['date', 'Symbol']).sort_index()

year = panel.index.get_level_values('date').year
train = panel.loc[(2003 <= year) & (year <= 2014)]
validation = panel.loc[(2015 <= year) & (year <= 2016)]
test = panel.loc[(2017 <= year) & (year <= 2019)]

In [10]:
def training_frame(symbol, mic):
    cal = xcals.get_calendar(mic, start='2003-01-01', end='2014-12-31')
    schedule = cal.schedule
    x = train.loc[train.index.get_level_values('Symbol') == symbol].droplevel('Symbol')
    x = x.assign(observed=True).reindex(schedule.index.rename('date'))
    x['observed'] = x.observed.eq(True)
    x['regular_session'] = ~x.index.isin(cal.early_closes.union(cal.late_opens))
    x['price'] = x.close_price.where(x.close_price.gt(0))
    x['daily_log_return'] = np.log(x.price).diff()
    x['daily_volatility'] = np.sqrt(x.rv5.where(x.rv5.ge(0)))
    return x

training = pd.concat({s: training_frame(s, mic) for s, mic in markets.items()}, names=['Symbol', 'date'])

In [11]:
full = training.loc[training.regular_session]
coverage = full.groupby(['Symbol', full.index.get_level_values('date').year.rename('year')]).agg(
    scheduled_days=('observed', 'size'), observed_days=('observed', 'sum'),
    price_days=('price', 'count'), rv5_days=('daily_volatility', 'count'))
coverage['rv5_coverage'] = coverage.rv5_days / coverage.scheduled_days
out = root / 'data/processed/european_indices'
out.mkdir(parents=True, exist_ok=True)
training.to_csv(out / 'train.csv')
coverage.to_csv(out / 'train_coverage.csv')
coverage

scheduled_days  observed_days  price_days  rv5_days  rv5_coverage
Symbol year                                                                   
.AEX   2003             253            250         250       250      0.988142
       2004             257            255         255       255      0.992218
       2005             257            257         257       257      1.000000
       2006             255            255         255       255      1.000000
       2007             253            253         253       253      1.000000
...                     ...            ...         ...       ...           ...
.SSMI  2010             254            254         254       254      1.000000
       2011             254            254         254       254      1.000000
       2012             250            250         250       250      1.000000
       2013             249            249         249       249      1.000000
       2014             249            249         249       249      1.000000

[156 rows x 5 columns]

## Decision: indices for the initial analysis

Use the eight indices with near-complete coverage over the 2003–2014 training period. Each is missing 3–8 scheduled full sessions in the `rv5` coverage counts, so none is literally complete. Defer the other five, each of which is missing 680 or more scheduled full sessions. The coverage assessment and exports still include all 13 indices.

| Initial use | Index | Country |
|---|---|---|
| Keep | AEX | Netherlands |
| Keep | BFX | Belgium |
| Keep | FCHI | France |
| Keep | FTSE | United Kingdom |
| Keep | GDAXI | Germany |
| Keep | IBEX | Spain |
| Keep | OSEAX | Norway |
| Keep | SSMI | Switzerland |
| Defer | BVLG | Portugal |
| Defer | FTMIB | Italy |
| Defer | OMXC20 | Denmark |
| Defer | OMXHPI | Finland |
| Defer | OMXSPI | Sweden |